<a href="https://colab.research.google.com/github/naitfarscape-ai/extrator-aic-fab/blob/main/extrator_aic_fab_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# --- CONFIGURAÇÃO E DEPENDÊNCIAS ---
import google.generativeai as genai
import pandas as pd
import numpy as np
from google.colab import files, userdata
import io
import re

def dms_to_dd(coord_str):
    """Converte coordenadas DMS (ex: 22º39'49"S) para Graus Decimais (DD)."""
    if not isinstance(coord_str, str): return None
    try:
        parts = re.split('[º\'" ]+', coord_str.strip())
        deg = float(parts[0])
        minu = float(parts[1])
        sec = float(parts[2])
        hem = parts[3].upper()
        dd = deg + minu/60 + sec/3600
        return -dd if hem in ['S', 'W'] else dd
    except:
        return None

# 1. SEGURANÇA: Obtém a chave dos 'Secrets' do Colab (ícone de chave no menu lateral)
try:
    CHAVE_API = userdata.get('GEMINI_API_KEY')
    genai.configure(api_key=CHAVE_API)
    model = genai.GenerativeModel('gemini-1.5-pro') # Pro recomendado para extração técnica
    print("--- SISTEMA DE VETORIZAÇÃO AIC PRONTO ---")
except:
    print("ERRO: Configure a 'GEMINI_API_KEY' nos Secrets do Colab.")

# 2. INTERFACE DE UPLOAD
print("Selecione os prints da AIC (PDF ou Imagens):")
uploaded = files.upload()

# 3. PROMPT ESTRUTURADO (ALINHADO AO MODELO TESTE.XLSX)
instrucoes = """
Atue como especialista em cartografia aeronáutica e GIS.
Analise as imagens da AIC e extraia os dados para preenchimento de uma planilha de eixos de REA/REH.

REGRAS DE EXTRAÇÃO E LÓGICA:
1. Trechos: Para 'n' fixos, gere 'n-1' trechos sequenciais.
2. Coluna 'Nome': Nome da rota (ex: ALFA).
3. Coluna 'Direção': Se houver altitudes diferentes por rumo, use 'Duas Direções'.
4. Coordenadas: Extraia no formato DMS (ex: 22º39'49"S).
5. Altitudes (Pés):
   - Se ranges diferentes por rumo: Preencha AltMaxA_to_B, AltMinA_to_B, AltMaxB_to_A, AltMinB_to_A.
   - Se range único: Preencha AltMax e AltMin.
   - Se altitude fixa: Preencha AltComp.
6. Atributos: Extraia 'Classe' (ex: G) e 'FCA' (ex: 135.675).
7. Geometria: Identifique Fixo_A e Fixo_B para cada trecho.

SAÍDA: Forneça apenas um JSON estruturado com a lista 'trechos' contendo as chaves:
Nome, Trecho, Fixo_A_Key, Fixo_A_DMS, Long_A_DMS, Fixo_B_Key, Lat_B_DMS, Long_B_DMS, RumoA_to_B, RumoB_to_A, AltMaxA_to_B, AltMinA_to_B, AltMaxB_to_A, AltMinB_to_A, AltMax, AltMin, AltComp, Classe, FCA.
"""

# 4. PROCESSAMENTO MULTIMODAL
imagens_payload = []
for nome, data in uploaded.items():
    imagens_payload.append({'mime_type': 'image/png', 'data': data})

if imagens_payload:
    print("IA interpretando eixos e fixos...")
    response = model.generate_content([instrucoes] + imagens_payload)

    # 5. TRATAMENTO DE DADOS E CONVERSÃO SIG
    try:
        # Extração limpa do JSON da resposta
        import json
        raw_text = response.text
        json_str = re.search(r'\[.*\]|\{.*\}', raw_text, re.DOTALL).group()
        data_json = json.loads(json_str)

        df = pd.DataFrame(data_json['trechos'] if 'trechos' in data_json else data_json)

        # Inserção de colunas para QGIS (Graus Decimais)
        df['Fixo_A_Lat'] = df['Lat_A_DMS'].apply(dms_to_dd)
        df['Fixo_A_Lon'] = df['Long_A_DMS'].apply(dms_to_dd)
        df['Fixo_B_Lat'] = df['Lat_B_DMS'].apply(dms_to_dd)
        df['Fixo_B_Lon'] = df['Long_B_DMS'].apply(dms_to_dd)

        # Semi_largura padrão (1.5 NM = 2778m)
        df['Semi_largura'] = 2778

        # Exportação final
        output_file = "AIC_VETORIZADA_QGIS.xlsx"
        df.to_excel(output_file, index=False)
        print(f"Sucesso! {len(df)} trechos processados.")
        files.download(output_file)
    except Exception as e:
        print(f"Erro no processamento: {e}")
        print("Resposta bruta da IA:", response.text)

--- SISTEMA DE CONVERSÃO DE REAs PRONTO ---
Selecione todas as imagens da REA (Parte 1, Parte 2, etc.)


TypeError: 'NoneType' object is not subscriptable